In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import json
from statistics import mean
import sys
sys.path.append("..")  # helpers.py лежит в корне курса
from helpers import add_user_message, chat, print_results

client = Anthropic()
model = "claude-haiku-5-5"

In [ ]:
def run_prompt(test_case):
    """Подставляет задачу в prompt и возвращает ответ Claude"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(client, model, messages)

In [ ]:
# Structured outputs вместо prefill: API возвращает JSON по схеме
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}
Solution: {output}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10
"""
    messages = []
    add_user_message(messages, eval_prompt)

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        output_config={
            "format": {
                "type": "json_schema",
                "schema": {
                    "type": "object",
                    "properties": {
                        "strengths": {
                            "type": "array",
                            "items": {"type": "string"},
                        },
                        "weaknesses": {
                            "type": "array",
                            "items": {"type": "string"},
                        },
                        "reasoning": {"type": "string"},
                        "score": {"type": "integer"},
                    },
                    "required": [
                        "strengths", "weaknesses", "reasoning", "score",
                    ],
                    "additionalProperties": False,
                },
            }
        },
    )
    text = next(
        block.text for block in response.content if block.type == "text"
    )
    return json.loads(text)

In [ ]:
def run_test_case(test_case):
    """Вызывает run_prompt и оценивает результат через grader"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }


def run_eval(dataset):
    """Запускает run_test_case для каждой задачи и печатает среднюю оценку"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    return results

In [ ]:
# dataset.json сохранили в уроке 11
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
for result in results:
    print("SCORE:", result["score"])
    print("REASONING:", result["reasoning"])
    print("-" * 80)

In [ ]:
print_results(results)